# Fraud Compliance Agent Notebook 01 — Plaid Sandbox source inventory

**Fraud Compliance Agent · Phase 0 · September 2026**  
**Status:** Completed source observation — findings pending mapping review  
**CRISP-DM phases:** Business understanding → Data understanding → Data preparation → Evaluation → Deployment  
**Decision supported:** P0-03 / ADR-003 — Plaid mapping and feature feasibility

---

## In plain English

This notebook is a **stocktake of the practice data** that Plaid makes available. Before building anything with bank-transaction information, we need to know what fields we can actually receive and which details are missing.

It is like checking the labels and contents of a box before deciding whether its parts can build the thing we need. It records safe observations about the Sandbox only; it does not make a fraud decision, train a model, or prove that a live bank connection will behave the same way.

Read the steps in order: prepare the safe practice connection, observe what returns, record only sanitised facts, then leave unknowns as unknowns for human review.

<a id="audience"></a>
## Who this notebook is for

This notebook is written for three audiences:

- **Fraud, product, and compliance reviewers** who need a clear answer to: what information can our proposed data source actually provide?
- **Data and risk engineers** who will turn approved source facts into a safe, canonical transaction format.
- **Technical reviewers** who need a repeatable record of the Sandbox observation without seeing credentials or raw financial data.

You do not need to understand the code to review this notebook. Every code cell is preceded by a plain-English explanation of what it does, what it deliberately does not do, and what a safe result means.

<a id="contents"></a>
## Contents

1. [What this notebook answers](#what-this-notebook-answers)
2. [How the Sandbox probe is kept safe](#how-the-sandbox-probe-is-kept-safe)
3. [Step 1: Prepare the local environment](#step-1--prepare-the-local-environment)
4. [Step 2: Wait for a practice-data response](#step-2--wait-for-a-practice-data-response)
5. [Step 3: Convert it into a safe inventory](#step-3--convert-it-into-a-safe-inventory)
6. [Step 4: Evaluate what we can and cannot conclude](#step-4--evaluate-what-we-can-and-cannot-conclude)
7. [Step 5: Produce a review artifact](#step-5--produce-a-review-artifact)
8. [What happens next](#what-happens-next)

<a id="what-this-notebook-answers"></a>
## What this notebook answers

The question is simple: **which transaction and source fields, including timestamp precision, are actually available in the selected Plaid Sandbox flow?**

A fraud system must not claim signals it cannot see. Before we decide which source facts or derived features can be used in a future risk decision, we need a factual inventory of Plaid's practice-data response.

This notebook runs four checks, in sequence:

1. **Local configuration check** — confirms the required credential names are configured without displaying their values.
2. **Sandbox readiness check** — creates a new practice-only test item, waits for its initial transaction pull within a fixed limit, and pages through available Sync updates.
3. **Schema inventory check** — records field paths, value types, nullability, timestamp-format observations, counts, and readiness metadata only.
4. **Review-artifact check** — writes a sanitised report that a reviewer can inspect before any mapping decision is made.

A successful run proves only that those observations were available in this Sandbox probe. It does **not** approve a canonical contract, a feature, a fraud policy, a model, or a production Plaid integration.

<a id="how-the-sandbox-probe-is-kept-safe"></a>
## How the Sandbox probe is kept safe

Plaid Sandbox is a practice environment, not a live customer account. Even so, this notebook treats provider responses as sensitive. Credentials are read from the ignored `apps/api/.env` file. The public token, access token, account references, and transaction values remain in memory and are never printed or committed. The report contains field paths, types, nullability, timestamp-precision observations, and response counts only.

### Non-goals

- This is not a production Plaid connector, transaction-ingestion job, or model-training notebook.
- It does not imply that a Sandbox response represents a live financial institution or historical replay.
- It does not silently convert missing data into `false`, `0`, or a guessed timestamp.
- It does not approve the canonical contract, feature set, risk policy, or model.


---

<a id="step-1--prepare-the-local-environment"></a>
## Step 1 — Prepare the local environment

### What this cell does

This cell finds the repository, loads the three Plaid configuration values from the ignored local API environment file, and confirms their **names** are present. It never prints their values. It also defines where the sanitised review report will be written.

### Why this matters

Keeping credentials outside the notebook and outside Git means the same notebook can be reviewed safely and repeated by an authorised developer. If configuration is incomplete, the notebook stops before making a network request.

**A successful result:** reports only that the three required configuration names are present and shows the report path.


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import subprocess
import time
from collections import defaultdict
from datetime import UTC, datetime
from pathlib import Path
from typing import Any
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display


def find_repository_root(start: Path) -> Path:
    """Find the project root from a local notebook working directory.

    Args:
        start: Current local directory; no provider data is read.

    Returns:
        The repository root containing the API environment example.

    Raises:
        RuntimeError: If the notebook is run outside this repository.

    Side effects:
        None.
    """
    for candidate in (start, *start.parents):
        if (candidate / 'apps' / 'api' / '.env.example').is_file():
            return candidate
    raise RuntimeError('Run this notebook from inside the fraud-compliance-agent repository.')


REPO_ROOT = find_repository_root(Path.cwd().resolve())
ENV_PATH = REPO_ROOT / 'apps' / 'api' / '.env'
REPORT_PATH = REPO_ROOT / 'docs' / 'proposals' / 'plaid-source-inventory.observation.json'

if not ENV_PATH.is_file():
    raise RuntimeError('Missing ignored apps/api/.env. Configure PLAID_CLIENT_ID, PLAID_SECRET, and PLAID_ENV locally.')

load_dotenv(ENV_PATH, override=False)
required_names = ('PLAID_CLIENT_ID', 'PLAID_SECRET', 'PLAID_ENV')
missing_names = [name for name in required_names if not os.getenv(name)]
if missing_names:
    raise RuntimeError('Missing required local configuration: ' + ', '.join(missing_names))

# Intentionally prints names only; never print values or the request payload.
print('Local configuration present: ' + ', '.join(required_names))
print(f'Report destination: {REPORT_PATH.relative_to(REPO_ROOT)}')


---

<a id="step-2--wait-for-a-practice-data-response"></a>
## Step 2 — Wait for a practice-data response

### What this cell does

This cell creates a fresh Sandbox-only practice item using Plaid's documented transaction-test user, exchanges its short-lived public token for an access token held only in memory, then polls Transactions Sync for up to 45 seconds. It follows all available pages when Plaid returns a paginated update. It does not use an existing customer or account.

### Why this matters

We need to observe the provider's actual response structure before writing a mapping. A generic failure message is used deliberately: raw provider errors can contain details that do not belong in notebook output or an issue tracker.

**A successful result:** confirms whether transaction fields became available before the fixed timeout. It does not display the response.


In [ ]:
PLAID_ENV = os.environ['PLAID_ENV'].strip().lower()
if PLAID_ENV != 'sandbox':
    raise RuntimeError('This discovery notebook is restricted to PLAID_ENV=sandbox.')

PLAID_BASE_URL = 'https://sandbox.plaid.com'


class PlaidProbeError(RuntimeError):
    def __init__(self, error_type: str, error_code: str):
        self.error_type = error_type
        self.error_code = error_code
        super().__init__(
            'Plaid Sandbox probe failed: ' + error_type + '/' + error_code + '. '
            'Inspect the Plaid Dashboard for request details without sharing provider payloads.'
        )


def plaid_post(path: str, payload: dict[str, Any]) -> dict[str, Any]:
    """Call one Sandbox endpoint while keeping credentials and responses in memory.

    Args:
        path: Relative Plaid Sandbox API path.
        payload: Non-secret request fields; credentials are added only from local environment variables.

    Returns:
        Decoded provider response for immediate in-memory schema inspection.

    Raises:
        PlaidProbeError: For a redacted provider error category.
        RuntimeError: For a network or malformed-response failure.

    Side effects:
        Sends a Sandbox request; never prints or persists credentials or raw responses.
    """
    request_payload = {
        'client_id': os.environ['PLAID_CLIENT_ID'],
        'secret': os.environ['PLAID_SECRET'],
        **payload,
    }
    request = Request(
        f'{PLAID_BASE_URL}{path}',
        data=json.dumps(request_payload).encode('utf-8'),
        headers={'Content-Type': 'application/json'},
        method='POST',
    )
    try:
        with urlopen(request, timeout=30) as response:
            return json.loads(response.read().decode('utf-8'))
    except HTTPError as exc:
        try:
            error_payload = json.loads(exc.read().decode('utf-8'))
            error_type = error_payload.get('error_type', 'unknown')
            error_code = error_payload.get('error_code', 'unknown')
        except (UnicodeDecodeError, json.JSONDecodeError):
            error_type, error_code = 'unknown', 'unknown'
        raise PlaidProbeError(error_type, error_code) from None
    except (URLError, TimeoutError, json.JSONDecodeError) as exc:
        raise RuntimeError('Plaid Sandbox network or response failure.') from exc


# `ins_109508` is Plaid's Sandbox-only test institution. The documented
# transaction-test user yields practice transaction history; neither value is
# a real customer or account identifier.
public_token_response = plaid_post(
    '/sandbox/public_token/create',
    {
        'institution_id': 'ins_109508',
        'initial_products': ['transactions'],
        'options': {
            'override_username': 'user_transactions_dynamic',
            'override_password': 'pass_good',
        },
    },
)

exchange_response = plaid_post(
    '/item/public_token/exchange',
    {'public_token': public_token_response['public_token']},
)

READINESS_TIMEOUT_SECONDS = 45
READINESS_POLL_SECONDS = 2
READY_STATUSES = {'INITIAL_UPDATE_COMPLETE', 'HISTORICAL_UPDATE_COMPLETE'}


MAX_PAGINATION_RESTARTS = 3


def read_sync_pages(access_token: str, cursor: str | None) -> tuple[list[dict[str, Any]], str | None, int]:
    """Read one complete Transactions Sync update without exposing its records.

    Args:
        access_token: In-memory Sandbox token; it must never be logged or persisted.
        cursor: Optional page-level checkpoint from the preceding Sync request.

    Returns:
        Response pages, the final page-level cursor, and pagination restart count.

    Raises:
        RuntimeError: If pagination repeatedly changes or lacks a required cursor.

    Side effects:
        Calls Plaid Sandbox only; raw pages remain in notebook memory.
    """
    for restart_count in range(MAX_PAGINATION_RESTARTS + 1):
        pages: list[dict[str, Any]] = []
        page_cursor = cursor
        try:
            while True:
                payload: dict[str, Any] = {'access_token': access_token}
                if page_cursor:
                    payload['cursor'] = page_cursor
                page = plaid_post('/transactions/sync', payload)
                pages.append(page)
                if not page.get('has_more', False):
                    return pages, page.get('next_cursor'), restart_count
                page_cursor = page.get('next_cursor')
                if not page_cursor:
                    raise RuntimeError('Plaid Sync pagination response lacked a next cursor.')
        except PlaidProbeError as exc:
            if exc.error_code != 'TRANSACTIONS_SYNC_MUTATION_DURING_PAGINATION':
                raise
    raise RuntimeError('Plaid Sync changed during pagination too often; retry the notebook later.')


sync_cursor: str | None = None
sync_pages: list[dict[str, Any]] = []
pagination_restart_count = 0
readiness_started = time.monotonic()
last_update_status = 'TRANSACTIONS_UPDATE_STATUS_UNKNOWN'
timed_out = False

while True:
    pages, sync_cursor, restarts = read_sync_pages(exchange_response['access_token'], sync_cursor)
    sync_pages.extend(pages)
    pagination_restart_count += restarts
    last_update_status = pages[-1].get('transactions_update_status', last_update_status)
    observed_transaction_count = sum(
        len(page.get(change_type, []))
        for page in sync_pages
        for change_type in ('added', 'modified', 'removed')
    )
    if last_update_status in READY_STATUSES and observed_transaction_count > 0:
        break
    if time.monotonic() - readiness_started >= READINESS_TIMEOUT_SECONDS:
        timed_out = True
        break
    time.sleep(READINESS_POLL_SECONDS)

inventory_response = {
    'added': [item for page in sync_pages for item in page.get('added', [])],
    'modified': [item for page in sync_pages for item in page.get('modified', [])],
    'removed': [item for page in sync_pages for item in page.get('removed', [])],
    'accounts': [item for page in sync_pages for item in page.get('accounts', [])],
    'has_more': False,
    'next_cursor': sync_cursor or '',
    'transactions_update_status': last_update_status,
}
readiness = {
    'status': last_update_status,
    'wait_seconds': round(time.monotonic() - readiness_started, 2),
    'page_count': len(sync_pages),
    'pagination_restart_count': pagination_restart_count,
    'timed_out': timed_out,
    'transaction_fields_observed': observed_transaction_count > 0,
}

# Do not display tokens, responses, or transaction values.
print('Sandbox probe completed; raw responses retained in memory only.')
print('Transaction fields observed before timeout: ' + str(readiness['transaction_fields_observed']))


---

<a id="step-3--convert-it-into-a-safe-inventory"></a>
## Step 3 — Convert it into a safe inventory

### What this cell does

This cell walks through the in-memory response and reduces it to a list of field paths, value types, nullability, cautious timestamp-format observations, and record counts. It never copies raw values into the report.

### Why this matters

A field called `date` does not prove that it contains a precise time, timezone, or point-in-time availability. The inventory therefore labels timestamp precision as something to inspect later rather than inventing an answer.

**A successful result:** displays a Pandas table of safe field metadata, then gives the number of field paths and added/modified/removed response counts. The table contains no transaction values.


In [ ]:
def collect_schema(value: Any, path: str = '$', inventory: dict[str, set[str]] | None = None) -> dict[str, set[str]]:
    """Collect field paths and types from an in-memory provider response.

    Args:
        value: Provider value to inspect; values themselves are never copied to the result.
        path: Current JSON-style field path.
        inventory: Optional accumulator of path-to-observed-type sets.

    Returns:
        Sanitised path/type inventory with no transaction, account, or credential values.

    Side effects:
        Mutates the supplied accumulator when one is provided.
    """
    inventory = inventory if inventory is not None else defaultdict(set)
    if value is None:
        inventory[path].add('null')
    elif isinstance(value, dict):
        inventory[path].add('object')
        for key, nested_value in value.items():
            collect_schema(nested_value, f'{path}.{key}', inventory)
    elif isinstance(value, list):
        inventory[path].add('array')
        for nested_value in value:
            collect_schema(nested_value, f'{path}[]', inventory)
    elif isinstance(value, bool):
        inventory[path].add('boolean')
    elif isinstance(value, int):
        inventory[path].add('integer')
    elif isinstance(value, float):
        inventory[path].add('number')
    elif isinstance(value, str):
        inventory[path].add('string')
    else:
        inventory[path].add(type(value).__name__)
    return inventory


def timestamp_precision(field_path: str, observed_types: list[str]) -> str:
    """Classify only evidenced timestamp precision without inferring missing semantics.

    Args:
        field_path: Sanitised schema path.
        observed_types: Types recorded for that path.

    Returns:
        A conservative precision classification for the review artifact.

    Side effects:
        None.
    """
    # A field name is evidence of neither timezone nor exact precision.
    field_name = field_path.rsplit('.', maxsplit=1)[-1].replace('[]', '').lower()
    timestamp_field_names = {
        'date', 'datetime', 'time', 'authorized_date', 'authorized_datetime',
        'posted_date', 'posted_datetime',
    }
    if field_name not in timestamp_field_names:
        return 'not_applicable'
    if observed_types != ['string']:
        return 'indeterminate'
    return 'inspect_format_before_mapping'


inventory = collect_schema(inventory_response)
schema_fields = [
    {
        'path': path,
        'observed_types': sorted(observed_types),
        'nullable': 'null' in observed_types,
        'timestamp_precision': timestamp_precision(path, sorted(observed_types)),
    }
    for path, observed_types in sorted(inventory.items())
]

transaction_counts = {
    key: len(inventory_response.get(key, []))
    for key in ('added', 'modified', 'removed')
}

# Render only sanitised schema metadata as a reviewable Pandas table.
schema_table = pd.DataFrame(schema_fields).rename(columns={
    'path': 'Plaid Sandbox field',
    'observed_types': 'Observed type(s)',
    'nullable': 'Can be empty',
    'timestamp_precision': 'Timestamp precision',
})
display(schema_table)

# Aggregate a small allowlist in memory; never display individual provider records.
analysis_columns = ['amount', 'pending', 'payment_channel', 'iso_currency_code']
analysis_frame = pd.DataFrame(inventory_response.get('added', [])).reindex(columns=analysis_columns)
if analysis_frame.empty:
    aggregate_analysis_table = pd.DataFrame(columns=['Pending', 'Payment channel', 'Currency', 'Transactions', 'Median absolute amount', 'Maximum absolute amount'])
else:
    # Treat amounts as source values only: do not infer canonical money or direction here.
    analysis_frame['absolute_source_amount'] = pd.to_numeric(analysis_frame['amount'], errors='coerce').abs()
    aggregate_analysis_table = (
        analysis_frame.groupby(['pending', 'payment_channel', 'iso_currency_code'], dropna=False)
        .agg(
            transactions=('amount', 'size'),
            median_absolute_source_amount=('absolute_source_amount', 'median'),
            maximum_absolute_source_amount=('absolute_source_amount', 'max'),
        )
        .reset_index()
        .rename(columns={
            'pending': 'Pending',
            'payment_channel': 'Payment channel',
            'iso_currency_code': 'Currency',
            'transactions': 'Transactions',
            'median_absolute_source_amount': 'Median absolute amount',
            'maximum_absolute_source_amount': 'Maximum absolute amount',
        })
    )
display(aggregate_analysis_table)

print(f'Inventory contains {len(schema_fields)} field paths.')
print(f'Response counts: {transaction_counts}')
print(f'Readiness status: {readiness["status"]}; waited {readiness["wait_seconds"]} seconds.')


---

<a id="step-4--evaluate-what-we-can-and-cannot-conclude"></a>
## Step 4 — Evaluate what we can and cannot conclude

### What this cell does

This cell packages the observation into a review report with the exact repository revision, run time, field inventory, response counts, and stated limitations. It creates a SHA-256 digest so the matching experiment record can identify the reviewed report precisely.

### What this result does not mean

A present field may still be unusable for the canonical contract if its business meaning, currency, timestamp precision, correction behaviour, or point-in-time availability is unresolved. A Sandbox observation never proves production availability or data freshness.

**A successful result:** says the inventory is observation-only and confirms that no canonical mapping has been approved.


In [ ]:
def git_revision(repository_root: Path) -> str:
    """Return a Git revision for reproducibility without failing a local probe.

    Args:
        repository_root: Repository directory in which Git is queried.

    Returns:
        Commit hash or an explicit unavailable marker.

    Side effects:
        Starts a read-only Git command.
    """
    try:
        return subprocess.check_output(
            ['git', 'rev-parse', 'HEAD'], cwd=repository_root, text=True, stderr=subprocess.DEVNULL
        ).strip()
    except (OSError, subprocess.CalledProcessError):
        return 'unavailable'


observation_status = (
    'observation_ready' if readiness['transaction_fields_observed']
    else 'observation_indeterminate'
)

report = {
    'artifact': 'plaid-source-inventory',
    'status': observation_status,
    'notebook': '01-plaid-sandbox-source-inventory.ipynb',
    'run_at': datetime.now(UTC).isoformat(),
    'git_revision': git_revision(REPO_ROOT),
    'source_system': 'plaid_sandbox',
    'environment_name': PLAID_ENV,
    'endpoint': '/transactions/sync',
    'readiness': readiness,
    'response_counts': transaction_counts,
    'field_inventory': schema_fields,
    'limitations': [
        'Sandbox availability does not prove production availability or freshness.',
        'Field names and types do not establish canonical semantics.',
        'Pending-to-posted, modification, removal, duplicate, and cursor behaviour require notebook 02.',
    ],
}

# A digest lets the experiment record refer to the report without storing raw provider data.
report_payload = json.dumps(report, sort_keys=True, indent=2)
report['report_sha256'] = hashlib.sha256(report_payload.encode('utf-8')).hexdigest()

print('Evaluation complete: inventory is observation-only; no canonical mapping is approved.')
print('Observation status: ' + observation_status)
print(f'Field paths recorded: {len(report["field_inventory"])}')


---

<a id="step-5--produce-a-review-artifact"></a>
## Step 5 — Produce a review artifact

### What this cell does

This cell writes the sanitised observation report to `docs/proposals/`. It writes no source values, tokens, identifiers, or raw payloads.

### Why this is called deployment

In CRISP-DM, deployment means putting a useful result into its intended process. For this discovery notebook, that means giving reviewers a precise, safe artifact to consider—not deploying a connector, model, policy, or API change.

**A successful result:** prints only the report digest. Review the generated file before linking it from the experiment record.


In [ ]:
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text(json.dumps(report, sort_keys=True, indent=2) + '\n', encoding='utf-8')

print('Sanitised report written for review.')
print(f'Report SHA-256: {report["report_sha256"]}')

# Before committing any generated report: inspect it, confirm it contains no raw values,
# link it from a completed docs/experiments record, and clear all notebook outputs.


---

<a id="what-happens-next"></a>
## What happens next

This notebook is the first observation only. Notebook 02 tests lifecycle behaviour such as pending-to-posted changes, corrections, duplicates, missing IDs, and cursor replay. Notebook 03 then proposes a mapping from the observed Plaid fields into our canonical transaction draft. Notebook 04 decides which potential fraud features are observed, derivable, unavailable, or simulated.

## Review checklist

- [ ] Credentials were present but never displayed.
- [ ] The report contains field names/types/counts only, with no raw source values.
- [ ] Each potentially useful field is marked observed, derivable, unavailable, or simulated in notebook 03/04—not inferred from this inventory alone.
- [ ] A matching experiment record contains the Git revision, report digest, findings, and limitations.
- [ ] This notebook's outputs are cleared before it is committed.
- [ ] Recommendation remains **proposed**: carry the observation into mapping review without approving a connector or feature.
